<a href="https://colab.research.google.com/github/Rheaxu/english-writing-sheet/blob/main/WritingPracticeSheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Configure Sentence

Enter the sentence you want to use for the writing practice sheet below:

### Configure the Sentence to Practice

This cell allows you to input the sentence that will be used to generate the writing practice sheet. The `sentence_input` variable holds the text you wish to trace.

In [8]:
# @title Sentence Input
sentence_input = 'I am Ambrose' # @param {type:"string"}

# ✏️ Writing Practice Sheet Generator

Stamps a **KG Primary Dots tracing sentence** onto Ms Sara's Writing Sheet and downloads a print-ready PDF.

**How to use — just 2 steps:**
1. Click **Runtime → Run all** (or Ctrl+F9) to set everything up
2. Scroll to the last cell, type your sentence, and click ▶ — the PDF downloads automatically

The KG Primary Dots font and the writing sheet template are embedded — no extra uploads needed.

In [9]:
# Install dependencies (runs once per Colab session)
!pip install pdf2image img2pdf Pillow -q
!apt-get install -y poppler-utils -q > /dev/null 2>&1
print("✅ Dependencies ready")

✅ Dependencies ready


### Core Generator Logic

This cell defines the `make_practice_sheet` function and embeds the necessary assets (the PDF template and the KG Primary Dots font) as base64 strings. The function handles text measurement, auto-scaling, double tracing for short sentences, and conversion to a landscape PDF format.

In [10]:
import base64, io, os, tempfile, numpy as np
from PIL import Image, ImageDraw, ImageFont
from pdf2image import convert_from_bytes
import img2pdf
from google.colab import files

# ── Embedded assets ─────────────────────────────────────────────────────
# Template PDF (MsSaraWritingSheet) and KG Primary Dots font are baked in.
# No uploads needed!

TEMPLATE_B64 = (
    "..."
)

FONT_B64 = (
    "..."
    "ABoAOwA+AB0AQQBBACEAQwBKACIATABPACo="
)

# Write font to a temp file PIL can read
_font_tmp = tempfile.NamedTemporaryFile(suffix='.ttf', delete=False)
_font_tmp.write(base64.b64decode(FONT_B64))
_font_tmp.close()
FONT_PATH = _font_tmp.name
print(f"\u2705 KG Primary Dots font ready at {FONT_PATH}")

# ── Layout constants (pixels at 150 dpi) ────────────────────────────────
DPI        = 150
BASELINE_Y = 377   # solid line = letter baseline (first writing row)
X_START    = 170   # left margin, starts after the arrow tip
RIGHT_M    = 40    # right margin
BASE_SIZE  = 175   # starting font size (auto-scaled down for long sentences)

# ── Sheet generator ─────────────────────────────────────────────────────
def make_practice_sheet(sentence: str) -> bytes:
    sentence = sentence.strip()
    if not sentence:
        raise ValueError('Sentence is empty — please type something!')

    # Decode template
    pages = convert_from_bytes(base64.b64decode(TEMPLATE_B64), dpi=DPI)
    base_img = pages[0]
    img_w, img_h = base_img.size
    avail_w = img_w - X_START - RIGHT_M

    # Load font at base size
    font_size = BASE_SIZE
    font = ImageFont.truetype(FONT_PATH, size=font_size)

    # Measure text width
    bbox = font.getbbox(sentence)
    text_w = bbox[2] - bbox[0]

    # Auto-scale down if sentence is too long for one line
    if text_w > avail_w:
        font_size = max(80, int(font_size * avail_w / text_w))
        font = ImageFont.truetype(FONT_PATH, size=font_size)
        bbox = font.getbbox(sentence)
        text_w = bbox[2] - bbox[0]

    # Position: ascent determines where to start drawing
    ascent, _ = font.getmetrics()
    draw_y = BASELINE_Y - ascent

    # Compose image
    result = base_img.copy()
    draw_obj = ImageDraw.Draw(result)
    draw_obj.text((X_START, draw_y), sentence, fill=(45, 45, 45), font=font)

    # If sentence fits in half the line, print it a second time
    half_w = avail_w // 2
    if text_w <= half_w:
        x2 = X_START + half_w + 15
        draw_obj.text((x2, draw_y), sentence, fill=(45, 45, 45), font=font)
        print(f'\u2139\ufe0f Short sentence — printed twice on the tracing line.')

    # Convert to landscape PDF (11 x 8.5 in)
    buf = io.BytesIO()
    result.save(buf, format='PNG', dpi=(DPI, DPI))
    return img2pdf.convert(
        buf.getvalue(),
        layout_fun=img2pdf.get_layout_fun(
            (img2pdf.in_to_pt(11), img2pdf.in_to_pt(8.5))
        )
    )

print("\u2705 Generator ready — scroll down to the next cell!")

✅ KG Primary Dots font ready at /tmp/tmp492afd_8.ttf
✅ Generator ready — scroll down to the next cell!


### Interactive PDF Generator Widget

This cell creates the user interface for the writing practice sheet generator. It includes a text input field for the sentence, a button to trigger PDF generation, and a status display. When the button is clicked, it calls the `make_practice_sheet` function and initiates the download of the generated PDF.

In [13]:
import ipywidgets as widgets
from IPython.display import display

# ── Interactive sentence input ────────────────────────────────────────
# Create a text input widget for the sentence, pre-filled with the configurable sentence_input.
txt = widgets.Text(
    value=sentence_input, # Use the configurable sentence_input
    placeholder='Type your sentence here...', # Placeholder text for the input field
    description='Sentence:', # Label for the input field
    layout=widgets.Layout(width='80%'), # Set the width of the input field
    style={'description_width': '80px'}, # Style the description label
)

# Create a button widget to trigger PDF generation.
btn = widgets.Button(
    description='Generate PDF ✏️', # Text displayed on the button
    button_style='success', # Visual style of the button (e.g., 'success', 'info', 'warning', 'danger')
    layout=widgets.Layout(width='160px'), # Set the width of the button
)

# Create a label widget to display status messages to the user.
status = widgets.Label(value='')

# Define the action to perform when the button is clicked.
def on_click(b):
    status.value = '⏳ Generating...' # Display a generating message
    try:
        # Generate the practice sheet PDF using the text from the input widget.
        pdf = make_practice_sheet(txt.value)
        fname = 'practice_sheet.pdf' # Define the filename for the generated PDF
        # Write the generated PDF bytes to a file.
        with open(fname, 'wb') as f:
            f.write(pdf)
        status.value = f"✅ Done! Downloading '{fname}'..." # Display a success message
        # Trigger the download of the generated PDF file.
        files.download(fname)
    except Exception as e:
        # If an error occurs, display an error message.
        status.value = f'❌ Error: {e}'

# Attach the on_click function to the button's click event.
btn.on_click(on_click)


# The Generator Button


In [12]:

# Display the text input widget, button, and status label in a vertical box layout.
display(widgets.VBox([txt, widgets.HBox([btn, status])]))